# actionability tester
same vibe as function_tester — just throw sentences at the classifier and see what comes back.  
no full pipeline, no CSV loading. just raw sentences → scores.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from nlp.preprocessing import clean_text
from nlp.actionability import (
    score_actionability_keywords,
    extract_srl_features,
    extract_named_entities,
    count_verb_tenses,
    assign_temporal_phase,
    _extract_spacy_features,
)

print('ready')

ready


## test sentences
swap these out freely — just match `lang` to the sentence language (`es` / `en` / `pt`)

In [2]:
# high actionability — imperative + spatial anchor + urgency
s1 = (
    "Evacúe inmediatamente la zona sur de Valencia. "
    "El puente Nou de la Campanar es peligroso. "
    "Llame al 112 si necesita rescate."
)

# medium — call to action but vague location
s2 = (
    "Las autoridades piden a los residentes que abandonen el área afectada "
    "antes del mediodía y se dirijan a los refugios habilitados."
)

# low — retrospective, no imperatives
s3 = (
    "Las inundaciones causaron graves daños en la región de Valencia "
    "durante el fin de semana pasado. El gobierno anunció un plan de ayuda."
)

# long-term / recovery — different actionability flavour
s4 = (
    "El gobierno destinará 200 millones para la reconstrucción de infraestructuras "
    "y la mitigación de futuros riesgos de inundación en zonas vulnerables."
)

# english sentence for comparison
s5 = (
    "Residents in the coastal district must evacuate immediately. "
    "Emergency shelters are open at the community center on River Road."
)

sentences = [
    (s1, 'es', 'high — imperative + spatial'),
    (s2, 'es', 'medium — vague location'),
    (s3, 'es', 'low — retrospective'),
    (s4, 'es', 'long-term / recovery'),
    (s5, 'en', 'english — high actionability'),
]

print(f'{len(sentences)} test sentences loaded')

5 test sentences loaded


## preprocessing check
just making sure `clean_text` doesn't mangle anything before passing to the scorer

In [3]:
for raw, lang, label in sentences:
    cleaned = clean_text(raw)
    changed = ' ← changed' if cleaned != raw else ''
    print(f"[{label}]")
    print(f"  {cleaned[:120]}{changed}")
    print()

[high — imperative + spatial]
  Evacúe inmediatamente la zona sur de Valencia. El puente Nou de la Campanar es peligroso. Llame al 112 si necesita resca

[medium — vague location]
  Las autoridades piden a los residentes que abandonen el área afectada antes del mediodía y se dirijan a los refugios hab

[low — retrospective]
  Las inundaciones causaron graves daños en la región de Valencia durante el fin de semana pasado. El gobierno anunció un 

[long-term / recovery]
  El gobierno destinará 200 millones para la reconstrucción de infraestructuras y la mitigación de futuros riesgos de inun

[english — high actionability]
  Residents in the coastal district must evacuate immediately. Emergency shelters are open at the community center on Rive



## keyword scores
this is the fast part — no spacy, just regex hits against the keyword lexicon in config

In [4]:
import pandas as pd

rows = []
for raw, lang, label in sentences:
    scores = score_actionability_keywords(clean_text(raw), lang)
    rows.append({'label': label, 'lang': lang, **scores})

kw_df = pd.DataFrame(rows).set_index('label')
kw_df

,lang,imperative_score,short_term_score,long_term_score,spatial_score,actionability_score
label,,,,,,
high — imperative + spatial,es,0,2,0,2,1.00
medium — vague location,es,0,0,0,1,0.20
low — retrospective,es,0,0,1,1,0.35
long-term / recovery,es,0,0,1,0,0.15
english — high actionability,en,1,2,0,2,1.35


## spacy: SRL + NER + tenses
one spacy pass per sentence — shows who did what where, entities, and verb tense breakdown

In [5]:
for raw, lang, label in sentences:
    feats = _extract_spacy_features(clean_text(raw), lang)
    print(f"[{label}]")
    print(f"  SRL   — agent={feats['has_agent']}  action={feats['has_action']}  location={feats['has_location']}  complete={feats['srl_complete']}")
    print(f"  tense — past={feats['past_tense']}  present={feats['present_tense']}  future={feats['future_tense']}")
    print(f"  locs  — {feats['top_locations']}")
    print(f"  orgs  — {feats['top_orgs']}")
    print()

[high — imperative + spatial]
  SRL   — agent=1  action=1  location=1  complete=1
  tense — past=0  present=3  future=0
  locs  — ["Valencia", "puente Nou de la Campanar"]
  orgs  — []

[medium — vague location]
  SRL   — agent=1  action=1  location=0  complete=0
  tense — past=0  present=3  future=0
  locs  — []
  orgs  — []

[low — retrospective]
  SRL   — agent=1  action=1  location=1  complete=1
  tense — past=2  present=0  future=0
  locs  — ["de Valencia"]
  orgs  — []

[long-term / recovery]
  SRL   — agent=1  action=1  location=0  complete=0
  tense — past=0  present=0  future=1
  locs  — []
  orgs  — []

[english — high actionability]
  SRL   — agent=1  action=1  location=1  complete=1
  tense — past=0  present=1  future=0
  locs  — ["River Road"]
  orgs  — []



## past tense penalty
the penalty lowers the score for articles that are mostly reporting past events — try it on s3 vs s1

In [6]:
PENALTY_WEIGHT = 0.15   # same as in actionability.py

for raw, lang, label in sentences:
    text = clean_text(raw)
    kw = score_actionability_keywords(text, lang)
    tenses = count_verb_tenses(text, lang)
    
    total_verbs = tenses['past_tense'] + tenses['present_tense'] + tenses['future_tense']
    ratio = tenses['past_tense'] / total_verbs if total_verbs > 0 else 0.0
    raw_score = kw['actionability_score']
    penalised = max(0, raw_score - PENALTY_WEIGHT * ratio)
    
    print(f"[{label}]")
    print(f"  raw={raw_score:.4f}  past_ratio={ratio:.2f}  penalised={penalised:.4f}")
    print()

[high — imperative + spatial]
  raw=1.0000  past_ratio=0.00  penalised=1.0000

[medium — vague location]
  raw=0.2000  past_ratio=0.00  penalised=0.2000

[low — retrospective]
  raw=0.3500  past_ratio=1.00  penalised=0.2000

[long-term / recovery]
  raw=0.1500  past_ratio=0.00  penalised=0.1500

[english — high actionability]
  raw=1.3500  past_ratio=0.00  penalised=1.3500



## temporal phase
assigning before / during / after relative to the flood onset date

In [7]:
import importlib
config = importlib.import_module('config.nlp_config')
flood_date = config.FLOOD_REFERENCE_DATE

# try different pub dates around the flood onset
test_dates = [
    ('2024-10-25', 'before onset'),
    ('2024-10-29', 'day of'),
    ('2024-11-02', 'during window'),
    ('2024-11-20', 'after'),
]

print(f'flood reference date: {flood_date}\n')
for pub_date, label in test_dates:
    phase = assign_temporal_phase(pub_date, flood_date)
    print(f'  {pub_date}  ({label})  →  {phase}')

flood reference date: 2024-10-29

  2024-10-25  (before onset)  →  before
  2024-10-29  (day of)  →  during
  2024-11-02  (during window)  →  during
  2024-11-20  (after)  →  after


## try your own sentence
paste any sentence here and change `lang` to `es`, `en`, or `pt`

In [8]:
my_text = "Prepárese para evacuar la zona costera — las autoridades prevén inundaciones severas esta noche."
my_lang = 'es'

text = clean_text(my_text)
kw    = score_actionability_keywords(text, my_lang)
feats = _extract_spacy_features(text, my_lang)

total_verbs = feats['past_tense'] + feats['present_tense'] + feats['future_tense']
ratio = feats['past_tense'] / total_verbs if total_verbs > 0 else 0.0
final = max(0, kw['actionability_score'] - 0.15 * ratio)

print(f'text:              {text}')
print(f'keyword score:     {kw["actionability_score"]:.4f}')
print(f'past tense ratio:  {ratio:.2f}')
print(f'final score:       {final:.4f}')
print()
print(f'imperative hits:   {kw["imperative_score"]}')
print(f'short-term hits:   {kw["short_term_score"]}')
print(f'long-term hits:    {kw["long_term_score"]}')
print(f'spatial hits:      {kw["spatial_score"]}')
print()
print(f'SRL complete:      {feats["srl_complete"]} (agent={feats["has_agent"]} action={feats["has_action"]} location={feats["has_location"]})')
print(f'locations found:   {feats["top_locations"]}')

text:              Prepárese para evacuar la zona costera — las autoridades prevén inundaciones severas esta noche.
keyword score:     0.5500
past tense ratio:  0.00
final score:       0.5500

imperative hits:   1
short-term hits:   0
long-term hits:    0
spatial hits:      1

SRL complete:      1 (agent=1 action=1 location=1)
locations found:   ["Prepárese"]
